# RQ2 --- Efficiency

In [1]:
import sys

sys.path.insert(0, "..")
import pandas as pd
from experiments._loader import load_all_results, success_only
from experiments._analysis import setup_matplotlib

setup_matplotlib()
df_all = load_all_results(include_baseline_fail=True)
df = success_only(df_all)
df["total_budget"] = (df["img_budget_used"] + df["txt_budget_used"]).clip(upper=df["budget_max"])
df["budget_utilization"] = df["total_budget"] / df["budget_max"]
# Loader uses recorded counts when available and the legacy 50-candidate estimate otherwise.

counts = df["model"].value_counts()
df = df[df["model"].isin(counts[counts >= 15].index)]


In [2]:
METRIC_COLS = [
    "img_budget_used",
    "txt_budget_used",
    "total_budget",
    "total_evaluations",
    "runtime",
]

SCENES = [
    ("MC", "multi"),
    ("SC-MI", "single/multi"),
    ("SC-SI", "single/solo"),
    ("Driving", "udacity"),
]


def format_metric(series: pd.Series) -> str:
    values = series.dropna()

    if values.empty:
        return "---"

    mean = values.mean()
    std = values.std()
    std = 0.0 if pd.isna(std) else std

    return f"${mean:.3f} \\pm {std:.3f}$"


records = []
for model in sorted(df["model"].dropna().unique()):
    for scene_label, obj_category in SCENES:
        scene_mask = (df["model"] == model) & (df["obj_category"] == obj_category)

        for genome_mode in ["multi", "image", "text"]:
            sub = df.loc[scene_mask & (df["genome_mode"] == genome_mode)]

            records.append(
                {
                    "model": model,
                    "scene": scene_label,
                    "genome_mode": genome_mode,
                    "n_opt": len(sub),
                    "evaluation_count_source": "recorded" if not sub.empty and sub["evaluation_count_source"].eq("recorded").all() else "estimated",
                    "img_budget": format_metric(sub["img_budget_used"]),
                    "txt_budget": format_metric(sub["txt_budget_used"]),
                    "total_budget": format_metric(sub["total_budget"]),
                    "total_evaluations": format_metric(sub["total_evaluations"]),
                    "runtime": format_metric(sub["runtime"]),
                }
            )

efficiency_df = pd.DataFrame(records)


In [3]:
efficiency_df.to_csv("efficiency.csv", index=False)


In [4]:
from experiments._analysis import MetricTable

In [5]:
MODEL_MAPPING = {
    "qwen": "Qwen3-VL",
    "kimi": "Kimi-VL",
    "intern": "InternVL3.5",
    "gemma": "Gemma3",
    "deepseek": "DeepseekVL2",
    "nemotron": "Nemotron3-NO",
}
efficiency = MetricTable(
    df=efficiency_df,
    title="Efficiency",
    model_mapping={k: v for k, v in MODEL_MAPPING.items() if k in efficiency_df["model"].unique()},
    scene_mapping={d: d for d in efficiency_df["scene"].unique()},
    metric_mapping={
        "n_opt": "n (optimized)",
        "img_budget": r"\faImage\ Budget $\downarrow$",
        "txt_budget": r"\faFont\ Budget $\downarrow$",
        "total_budget": r"$\sum$ Budget $\downarrow$",
        "total_evaluations": r"\# SUT Eval $\downarrow$",
        "evaluation_count_source": "Eval count",
        "runtime": r"Runtime (sec) $\downarrow$",
    },
)


In [6]:
print(efficiency)

\begin{tabular}{llccccccc}
\toprule
Model & Scene & n (optimized) & \faImage\ Budget $\downarrow$ & \faFont\ Budget $\downarrow$ & $\sum$ Budget $\downarrow$ & \# SUT Eval $\downarrow$ & Eval count & Runtime (sec) $\downarrow$ \\
\midrule
\multirow{12}{*}{Qwen3-VL} & \multirow{3}{*}{MC} & 80 & $0.196 \pm 0.150$ & $0.405 \pm 0.161$ & $0.601 \pm 0.218$ & $458.125 \pm 1079.922$ & estimated & $33.934 \pm 62.947$ \\
 &  & 80 & $0.219 \pm 0.136$ & $0.000 \pm 0.000$ & $0.219 \pm 0.136$ & $3003.125 \pm 2348.907$ & estimated & $193.700 \pm 162.882$ \\
 &  & 80 & $0.000 \pm 0.000$ & $0.439 \pm 0.149$ & $0.439 \pm 0.149$ & $558.125 \pm 1193.719$ & estimated & $30.002 \pm 49.836$ \\
\cmidrule(lr){2-9}
 & \multirow{3}{*}{SC-MI} & 94 & $0.186 \pm 0.149$ & $0.440 \pm 0.162$ & $0.626 \pm 0.217$ & $812.234 \pm 1363.201$ & estimated & $61.045 \pm 92.717$ \\
 &  & 94 & $0.206 \pm 0.120$ & $0.000 \pm 0.000$ & $0.206 \pm 0.120$ & $2996.277 \pm 2340.127$ & estimated & $158.312 \pm 137.089$ \\
 &  & 94 & $0.